# **Phase - 1**

## **STEP-0: EXECUTION**

In [ ]:
BASE_PATH = "/kaggle/input/datasets/gunadeepchennupati/dentex-processed-data/processed_dataset"

### **LOAD DATASET**

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder(BASE_PATH + "/train", transform=transform)
val_dataset = datasets.ImageFolder(BASE_PATH + "/val", transform=transform)

### **CHECK CLASS MAPPING**

In [ ]:
print("Class mapping:", train_dataset.class_to_idx)

### **CREATE DATALOADER**

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

### **CHECK BATCH**

In [ ]:
images, labels = next(iter(train_loader))

print("Image shape:", images.shape)
print("Labels:", labels[:10])

### **VISUAL CHECK**

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(images[3].permute(1, 2, 0))
plt.title(f"Label: {labels[0].item()}")
plt.axis('off')
plt.show()

### **SIZE CHECK**

In [ ]:
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

In [ ]:
from collections import Counter

labels_list = [label for _, label in train_dataset]
print(Counter(labels_list))

## **STEP-1: TRANSFORMS**

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

### **TRANSFORMS**

In [ ]:
from torchvision import transforms

transform = transforms.Compose([      
    transforms.Resize((224, 224)),  # enforce consistent size
    transforms.ToTensor(),          # convert to tensor
    transforms.Normalize(           # REQUIRED for pretrained ResNet
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

### **DATASET LOAD**

In [ ]:
from torchvision import datasets

train_dataset = datasets.ImageFolder(
    BASE_PATH + "/train",
    transform=transform
)

val_dataset = datasets.ImageFolder(
    BASE_PATH + "/val",
    transform=transform
)

### **DATALOADER**

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

### **VERIFICATION**

In [ ]:
images, labels = next(iter(train_loader))

print("Shape:", images.shape)
print("Min:", images.min().item())
print("Max:", images.max().item())

### **VISUAL CHECK**

In [ ]:
import matplotlib.pyplot as plt

# Unnormalize
img = images[3].permute(1, 2, 0).numpy()
img = (img * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]

plt.imshow(img)
plt.title(f"Label: {labels[0].item()}")
plt.axis('off')
plt.show()

## **STEP-2: RESNET18**

### IMPORTS + DEVICE

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

### LOAD PRETRAINED RESNET18

In [ ]:
model = models.resnet18(pretrained=True)

### MODIFY FINAL LAYER

In [ ]:
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

### MOVE MODEL TO GPU

In [ ]:
model = model.to(device)

### LOSS FUNCTION

In [ ]:
# Class weights based on your dataset
weights = torch.tensor([
    0.6376942537043729,   # caries (class 0)
    2.3156167979002626    # no_caries (class 1)
]).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)

### OPTIMIZER

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
print(model.fc)

In [ ]:
images, labels = next(iter(train_loader))

images, labels = images.to(device), labels.to(device)

print(images.device)
print(labels.device)

## **STEP-3: TRAINING LOOP**

### TRAIN FUNCTION

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc

### VALIDATION FUNCTION

In [ ]:
def validate(model, loader, criterion, device):
    model.eval()
    
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    from sklearn.metrics import f1_score
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / total
    epoch_f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    
    return epoch_loss, epoch_acc, epoch_f1_macro

### TRAINING LOOP

In [ ]:
num_epochs = 10

best_val_f1 = 0.0

for epoch in range(num_epochs):
    
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    
    val_loss, val_acc, val_f1_macro = validate(model, val_loader, criterion, device)
    
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f} | Val F1 (macro): {val_f1_macro:.4f}")
    
    # Save best model — checkpoint on macro-F1, not raw accuracy (T0.3,
    # docs/research/MASTER_PLAN_2026H2.md §5 W2). Accuracy rewards the majority
    # class on this imbalanced gate; macro-F1 doesn't.
    if val_f1_macro > best_val_f1:
        best_val_f1 = val_f1_macro
        torch.save(model.state_dict(), "best_model.pth")
        print("🔥 Best model saved!")
    
    print("-" * 40)

## **STEP-4: EVALUATION**

### LOAD BEST MODEL

In [ ]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

### COLLECT PREDICTIONS

In [ ]:
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

### METRICS

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=["caries", "no_caries"]))

**why no_caries recall is lower?**
* CLASS IMBALANCE
* VISUAL SIMILARITY
* LOSS DESIGN TRADE-OFF

## **Step-5: Save + Final validation**

### SAVE FINAL MODEL

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'class_to_idx': train_dataset.class_to_idx
}, "final_model.pth")

### LOAD MODEL

In [ ]:
checkpoint = torch.load("final_model.pth")

model.load_state_dict(checkpoint['model_state_dict'])
class_to_idx = checkpoint['class_to_idx']

model = model.to(device)
model.eval()

print("Loaded mapping:", class_to_idx)

### INFERENCE FUNCTION

In [ ]:
from PIL import Image

def predict_image(image_path, model, transform, device, idx_to_class):
    img = Image.open(image_path).convert("RGB")
    img = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(img)
        _, pred = torch.max(output, 1)
    
    pred_idx = pred.item()
    pred_label = idx_to_class[pred_idx]
    
    return pred_idx, pred_label

### TEST PREDICTION

In [ ]:
test_img = "/kaggle/input/datasets/gunadeepchennupati/dentex-processed-data/processed_dataset/val/caries/13_49.png"

idx_to_class = {v: k for k, v in class_to_idx.items()}

pred_idx, pred_label = predict_image(test_img, model, transform, device, idx_to_class)

print(f"Prediction → {pred_label} (class {pred_idx})")

In [ ]:
test_img = "/kaggle/input/datasets/gunadeepchennupati/dentex-processed-data/processed_dataset/val/no_caries/16_55.png"

idx_to_class = {v: k for k, v in class_to_idx.items()}

pred_idx, pred_label = predict_image(test_img, model, transform, device, idx_to_class)

print(f"Prediction → {pred_label} (class {pred_idx})")

## **MINIMAL SETUP**

### IMPORTS + DEVICE

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### TRANSFORMS

In [ ]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

### LOAD MODEL

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=None)

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)  # temporary

checkpoint = torch.load("final_model.pth")

model.load_state_dict(checkpoint['model_state_dict'])
class_to_idx = checkpoint['class_to_idx']

model = model.to(device)
model.eval()

print("Loaded mapping:", class_to_idx)

# **Phase - 2**

## **STEP-1: IMPORTS**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import resnet18
from torch.utils.data import DataLoader

## **STEP-2: DEVICE**

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## **STEP-3: TRANSFORMS**

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## **STEP-4: LOAD DATA**

In [4]:
BASE_PATH = "/kaggle/input/datasets/gunadeepchennupati/dentex-severity-processed-dataset/severity_data"

train_dataset = datasets.ImageFolder(f"{BASE_PATH}/train", transform=transform)
val_dataset   = datasets.ImageFolder(f"{BASE_PATH}/val", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(train_dataset.class_to_idx)

{'caries': 0, 'deep_caries': 1}


## **STEP-5: LOAD PHASE-1 MODEL**

In [7]:
from torchvision.models import resnet18
import torch.nn as nn

model = resnet18(weights=None)

# Step 1: Match Phase-1 architecture
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

# Step 2: Load Phase-1 weights
checkpoint = torch.load("/kaggle/working/final_model.pth", map_location=device)

if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    model.load_state_dict(checkpoint)

# Step 3: Reset FC for Phase-2 (CRITICAL)
model.fc = nn.Linear(num_features, 2)

# Step 4: Move to device
model = model.to(device)

## **STEP-6: FREEZE BACKBONE**

In [8]:
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

## **STEP-7: CLASS WEIGHTS**

In [12]:
weights = torch.tensor([0.5, 3.5]).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

In [ ]:
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

## **STEP-8: TRAIN / VAL FUNCTIONS**

In [7]:
def train_one_epoch(model, loader):
    model.train()
    total_loss, correct = 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

In [ ]:
def validate(model, loader):
    model.eval()
    total_loss, correct = 0, 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    from sklearn.metrics import f1_score

    val_f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return total_loss / len(loader), correct / len(loader.dataset), val_f1_macro

## **STEP-9: TRAIN FC ONLY**

In [ ]:
best_val_f1 = 0

for epoch in range(5):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc, val_f1_macro = validate(model, val_loader)

    print(f"Epoch {epoch+1}")
    print(f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val F1 (macro): {val_f1_macro:.4f}")

    # SAVE BEST MODEL — checkpoint on macro-F1 (T0.3), not raw accuracy
    if val_f1_macro > best_val_f1:
        best_val_f1 = val_f1_macro
        torch.save(model.state_dict(), "best_phase2_stage1.pth")
        print("Best model saved")

## **STEP-10: UNFREEZE**

In [13]:
for param in model.parameters():
    param.requires_grad = True

In [14]:
# optimizer = optim.Adam(model.parameters(), lr=5e-5)

## **STEP-11: FULL FINE-TUNING**

In [ ]:
for epoch in range(10):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc, val_f1_macro = validate(model, val_loader)

    print(f"[Fine-tune] Epoch {epoch+1}")
    print(f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val F1 (macro): {val_f1_macro:.4f}")

    if val_f1_macro > best_val_f1:
        best_val_f1 = val_f1_macro
        torch.save(model.state_dict(), "best_phase2_final.pth")
        print("Best model saved")

## **EVALUATION STAGE**

### LOAD BEST MODEL

In [6]:
model.load_state_dict(torch.load("best_phase2_final.pth"))
model = model.to(device)

In [5]:
model.load_state_dict(torch.load("best_phase2_final.pth"))
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

### CONFUSION MATRIX + REPORT

In [18]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=['caries','deep_caries']))

Confusion Matrix:
 [[87 14]
 [13 19]]

Classification Report:

              precision    recall  f1-score   support

      caries       0.87      0.86      0.87       101
 deep_caries       0.58      0.59      0.58        32

    accuracy                           0.80       133
   macro avg       0.72      0.73      0.73       133
weighted avg       0.80      0.80      0.80       133



In [3]:
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [5]:
BASE_PATH = "/kaggle/input/datasets/gunadeepchennupati/dentex-severity-processed-dataset/severity_data"

train_dataset = datasets.ImageFolder(f"{BASE_PATH}/train", transform=transform)
val_dataset   = datasets.ImageFolder(f"{BASE_PATH}/val", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Class mapping:", train_dataset.class_to_idx)

Class mapping: {'caries': 0, 'deep_caries': 1}


In [6]:
model = resnet18(weights=None)

# match Phase-2 structure
model.fc = nn.Linear(model.fc.in_features, 2)

#  LOAD BEST MODEL FROM PREVIOUS TRAINING
model.load_state_dict(torch.load("/kaggle/working/best_phase2_final.pth"))

model = model.to(device)
model.eval()

print("Model loaded successfully")

Model loaded successfully


In [13]:
weights = torch.tensor([0.5, 3.5]).to(device)



criterion = nn.CrossEntropyLoss(weight=weights)

In [8]:
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=3e-5)

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    total, correct, loss_sum = 0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        loss_sum += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return loss_sum / len(loader), correct / total


def validate(model, loader):
    model.eval()
    total, correct, loss_sum = 0, 0, 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss_sum += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    from sklearn.metrics import f1_score

    val_f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return loss_sum / len(loader), correct / total, val_f1_macro

In [ ]:
best_val_f1 = 0

for epoch in range(9):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc, val_f1_macro = validate(model, val_loader)

    print(f"[Fix Training] Epoch {epoch+1}")
    print(f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val F1 (macro): {val_f1_macro:.4f}")

    if val_f1_macro > best_val_f1:
        best_val_f1 = val_f1_macro
        torch.save(model.state_dict(), "best_phase2_fixed.pth")
        print("Best model saved")

In [12]:
model.load_state_dict(torch.load("best_phase2_fixed.pth"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=['caries','deep_caries']))

Confusion Matrix:
 [[77 24]
 [ 7 25]]

Classification Report:

              precision    recall  f1-score   support

      caries       0.92      0.76      0.83       101
 deep_caries       0.51      0.78      0.62        32

    accuracy                           0.77       133
   macro avg       0.71      0.77      0.72       133
weighted avg       0.82      0.77      0.78       133



In [14]:
weights = torch.tensor([0.6, 3.0]).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)

In [15]:
model.load_state_dict(torch.load("best_phase2_fixed.pth"))
model = model.to(device)

In [16]:
optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)

In [ ]:
# Resuming from best_phase2_fixed.pth. Previously this threshold was a hardcoded
# accuracy (0.8195) copied from the "Best model saved" printout of the epoch this
# checkpoint came from (cell 113, epoch 8). Since checkpointing now uses macro-F1
# (T0.3), the equivalent starting point is the macro-F1 of that same checkpoint —
# taken from cell 114's already-saved classification_report ("macro avg" f1-score
# = 0.72; 2dp is all classification_report prints, so that's the available
# precision). VERY IMPORTANT: if resuming from a different checkpoint in future,
# recompute this via validate() rather than hand-copying a printed report.
best_val_f1 = 0.72

for epoch in range(3):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc, val_f1_macro = validate(model, val_loader)

    print(f"[Fix Training] Epoch {epoch+1}")
    print(f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val F1 (macro): {val_f1_macro:.4f}")

    if val_f1_macro > best_val_f1:
        best_val_f1 = val_f1_macro
        torch.save(model.state_dict(), "best_phase2_fixed2.pth")
        print("Best model saved")

In [18]:
model.load_state_dict(torch.load("best_phase2_fixed2.pth"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=['caries','deep_caries']))

Confusion Matrix:
 [[81 20]
 [ 5 27]]

Classification Report:

              precision    recall  f1-score   support

      caries       0.94      0.80      0.87       101
 deep_caries       0.57      0.84      0.68        32

    accuracy                           0.81       133
   macro avg       0.76      0.82      0.77       133
weighted avg       0.85      0.81      0.82       133



In [19]:
torch.save({
    'model_state_dict': model.state_dict(),
    'class_to_idx': train_dataset.class_to_idx
}, "phase2_final_bundle.pth")

## **Phase 2b — Arm B: Augmentation Ablation (T0.1)**

Per `docs/results/METHODOLOGY_CHANGELOG.md` (2026-08-15, T0.1): `transform_train` /
`transform_val` were defined two cells above (right before **STEP-4: LOAD DATA** was re-run
for the "Fix Training" stages) but never actually used — everything above this point
(**Arm A**, cells building up to `best_phase2_fixed2.pth`) trained and evaluated on the
plain, unaugmented transform the whole time. This section reruns the *same* two-stage
fine-tune recipe — same starting checkpoint (`best_phase2_final.pth`), same class-weight
schedule (`[0.5, 3.5]` → `[0.6, 3.0]`), same learning rate (3e-5), same epoch counts (9 then
3), same macro-F1 checkpointing (T0.3) — with only one change: the training set actually
gets `transform_train` applied this time. `val` does not change between arms —
`transform_val` is pixel-for-pixel the same resize+normalize pipeline as the plain
`transform` used everywhere else (standard practice: augment train only, not val), so the
existing `val_loader` is reused unchanged for both arms.

**Report both arms side by side. Do not treat Arm B as a replacement for Arm A and discard
the comparison** — the entire point is the delta between them, not picking a new best.

In [ ]:
# Arm B dataset — same DENTEX severity data. transform_val is identical to the plain
# `transform` used for val elsewhere (resize+normalize, no augmentation — standard
# practice is to augment train only), so `val_loader` is reused unchanged for both arms.
# Only the TRAIN set actually gets transform_train applied here.
train_dataset_aug = datasets.ImageFolder(f"{BASE_PATH}/train", transform=transform_train)
train_loader_aug = DataLoader(train_dataset_aug, batch_size=32, shuffle=True)

print("Arm B (augmented) train size:", len(train_dataset_aug))
print("Val size (shared with Arm A):", len(val_dataset))

### Arm B — Stage 1 (mirrors Arm A's cells 109–113: `[0.5, 3.5]`, lr 3e-5, 9 epochs)

In [ ]:
model_aug = resnet18(weights=None)
model_aug.fc = nn.Linear(model_aug.fc.in_features, 2)
model_aug.load_state_dict(torch.load("/kaggle/working/best_phase2_final.pth"))
model_aug = model_aug.to(device)

for param in model_aug.parameters():
    param.requires_grad = True

weights = torch.tensor([0.5, 3.5]).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.Adam(model_aug.parameters(), lr=3e-5)

best_val_f1_aug = 0

for epoch in range(9):
    train_loss, train_acc = train_one_epoch(model_aug, train_loader_aug)
    val_loss, val_acc, val_f1_macro = validate(model_aug, val_loader)

    print(f"[Arm B, Stage 1] Epoch {epoch+1}")
    print(f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val F1 (macro): {val_f1_macro:.4f}")

    if val_f1_macro > best_val_f1_aug:
        best_val_f1_aug = val_f1_macro
        torch.save(model_aug.state_dict(), "best_phase2_augmented_stage1.pth")
        print("Best model saved")

### Arm B — Stage 2 (mirrors Arm A's cells 115–118: `[0.6, 3.0]`, lr 3e-5, 3 more epochs)

Unlike Arm A's cell 118, no hand-copied resume threshold is needed here — `best_val_f1_aug`
carries over from the Stage 1 cell above in the same live kernel session, so there's nothing
to recover from a printed report.

In [ ]:
model_aug.load_state_dict(torch.load("best_phase2_augmented_stage1.pth"))
model_aug = model_aug.to(device)

weights = torch.tensor([0.6, 3.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.Adam(model_aug.parameters(), lr=3e-5)

for epoch in range(3):
    train_loss, train_acc = train_one_epoch(model_aug, train_loader_aug)
    val_loss, val_acc, val_f1_macro = validate(model_aug, val_loader)

    print(f"[Arm B, Stage 2] Epoch {epoch+1}")
    print(f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val F1 (macro): {val_f1_macro:.4f}")

    if val_f1_macro > best_val_f1_aug:
        best_val_f1_aug = val_f1_macro
        torch.save(model_aug.state_dict(), "best_phase2_augmented.pth")
        print("Best model saved")

### Arm A vs Arm B — side-by-side comparison (val only)

Both checkpoints are reloaded fresh here (rather than trusting whatever `model`/`model_aug`
happen to hold in memory at this point) so this cell is self-contained and re-runnable on
its own. This is a **val-only** comparison on the pre-T0.2 split — once
`scripts/dentex/split_dentex_holdout.py` has actually been run against the real data, re-run
this same comparison against the new `val/` **and** report both arms on the locked `test/`
split too (T0.2's "two-arm run" requirement).

In [ ]:
from sklearn.metrics import f1_score


def evaluate_checkpoint(checkpoint_path, loader):
    """Load a checkpoint into a fresh model and return (preds, labels) on the given loader."""
    m = resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, 2)
    m.load_state_dict(torch.load(checkpoint_path))
    m = m.to(device)
    m.eval()

    preds, labels_out = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = m(images)
            _, p = torch.max(outputs, 1)
            preds.extend(p.cpu().numpy())
            labels_out.extend(labels.numpy())
    return preds, labels_out


preds_a, labels_a = evaluate_checkpoint("best_phase2_fixed2.pth", val_loader)
f1_a = f1_score(labels_a, preds_a, average='macro', zero_division=0)

preds_b, labels_b = evaluate_checkpoint("best_phase2_augmented.pth", val_loader)
f1_b = f1_score(labels_b, preds_b, average='macro', zero_division=0)

print("=" * 60)
print("ARM COMPARISON — Val macro-F1 (T0.1 augmentation ablation)")
print("=" * 60)
print(f"Arm A (plain, no augmentation): {f1_a:.4f}")
print(f"Arm B (augmented):              {f1_b:.4f}")
print(f"Delta (B - A):                  {f1_b - f1_a:+.4f}")
print()
print("Arm A classification report:")
print(classification_report(labels_a, preds_a, target_names=['caries', 'deep_caries']))
print("Arm B classification report:")
print(classification_report(labels_b, preds_b, target_names=['caries', 'deep_caries']))
print()
print("NOTE: this is the pre-T0.2 val/ split for both arms. Once")
print("scripts/dentex/split_dentex_holdout.py has been run for real, re-run this cell")
print("against the new val/, AND report both arms on the locked test/ split too.")

SAVE TRANSFORM, We Used
* Resize(224)
* Normalize(mean, std)